# Formas de onda y controles de primeras llegadas

**Nivel 1 — Fundamentos**  
**Prerrequisitos:** nociones de Python; 35 min.

## Objetivos

Siga la evidencia desde el contrato fuente hasta el resultado. Declare unidades y supuestos, reproduzca el cálculo e identifique qué inferencia no respalda la evidencia.

## Preparación

Usa módulos y registros versionados desde la raíz descubierta; no descarga ni modifica fuentes. Instale versiones exactas: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Datos

Hay trazas STEAD retenidas y recibos FWI; distinga conteos medidos, FWI sintético, etiquetas humanas y activos web.

## Método

Una traza muestreada es `x_c[n]` en `t=nΔt`; el subconjunto STEAD usa 100 Hz, 6.000 muestras, 60 s y tres canales en orden componente-mayor. Son conteos instrumentales normalizados sin restitución, no velocidad del suelo; picks P/S son referencias de analistas, no verdad geológica. M08 usa `classical_stalta_picks`: ventanas de energía corta/larga `N=redondear(T/Δt)` y cruces de umbral detectan inicios candidatos; el segundo se llama S solo provisionalmente. Grafique una traza retenida real con referencias y picks para ver el desacuerdo temporal. STA/LTA detecta inicios, no clasifica fases; las etiquetas no entran en la detección.

A 100 muestras/s, Nyquist es 50 Hz; respuesta instrumental y preprocesamiento determinan ancho de banda útil. Rayos tomográficos dependen de velocidad actual; FWI usa fase y amplitud completas. Mantenga trazas/disparos retenidos fuera del ajuste. Menor residuo de entrenamiento puede coexistir con peor WRMS retenido o estructura en ciclo equivocado.

## Resultados

Lea ejes físicos y unidades; distinga ajuste, error contra verdad sintética y acuerdo con etiquetas humanas. El recálculo debe concordar con el recibo. Un modelo graficado no demuestra unicidad, transferencia a campo ni resolución.

### Diagnostic 1

Inspeccione el recibo y procedencia del caso.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Load one real manifested trace and run the fixed product STA/LTA baseline.
import json,sys
from pathlib import Path
manifest=json.loads((root/"data/derived/phase/browser-assets/stead/manifest.json").read_text(encoding="utf-8")); rec=manifest["records"][0]; folder=root/"data/derived/phase/browser-assets/stead"; trace=np.fromfile(folder/rec["waveform_file"],dtype="<f4").reshape(manifest["input"]["shape"])[0].T
sys.path.insert(0,str(root/"data-pipeline")); from phase_picking import classical_stalta_picks
b=json.loads((root/"data/derived/phase/stead-heldout-benchmark.json").read_text(encoding="utf-8")); p=b["classical_parameters"]; dt=1/manifest["input"]["sample_rate_hz"]; pick=classical_stalta_picks(trace,sample_interval_s=dt,threshold=p["trigger_ratio"],sta_s=p["STA_s"],lta_s=p["LTA_s"],minimum_ps_s=p["minimum_PS_s"])
print(rec["trace_id"],trace.shape,manifest["input"]["unit"],"analyst",rec["analyst_p_s"],rec["analyst_s_s"],"STA/LTA",pick)

### Diagnostic 2

Calcule la comparación y grafíquela con unidades físicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot real normalized components, analyst references and provisional detector markers.
import matplotlib.pyplot as plt
t=np.arange(len(trace))*dt; fig,ax=plt.subplots(3,1,figsize=(9,6),sharex=True,constrained_layout=True)
for i,n in enumerate(manifest["input"]["component_order"]): ax[i].plot(t,trace[:,i],lw=.55); ax[i].set_ylabel(n+" (norm. counts)"); ax[i].grid(alpha=.2)
for a in ax:
 a.axvline(rec["analyst_p_s"],ls="--",color="crimson",label="analyst P"); a.axvline(rec["analyst_s_s"],ls="--",color="purple",label="analyst S")
 for q,c in [("P","#e07a24"),("S","#3b7a57")]:
  if pick[q] is not None: a.axvline(pick[q],color=c,label="M08 provisional "+q)
ax[0].legend(ncol=2,fontsize=8); ax[-1].set_xlabel("Time (s)"); ax[0].set_title("Real held-out trace · analyst labels are not truth"); plt.show()

## Resumen

La conclusión depende de geometría, adquisición, ruido, partición, parametrización y derechos de fuente. Conserve negativos y nombre la observación o control siguiente.

## Referencias

- [https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html)
- [https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/](https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/)
- [https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html](https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html)